# M6 — Integration Capstone

> 把 M1~M5 全部串起來，做出一個**可以放履歷**的 ETL pipeline

## 專案：電商訂單 ETL

```
┌────────────────────────────────────────────────────────┐
│                    ETL Pipeline                         │
│                                                         │
│  ┌────────┐   ┌───────────┐   ┌──────────────────┐    │
│  │Extract │→→ │ Transform │→→ │       Load       │    │
│  └────────┘   └───────────┘   └──────────────────┘    │
│     │              │                    │              │
│  CSV + SQL     清理+合併+KPI        SQLite + Excel      │
│   (M1+M2)       (Pandas)              (M1+M2)          │
│                                                         │
│  整個過程全程 log（M3），參數從 config 讀（M4），        │
│  所有函式有 type hints（M5）                            │
└────────────────────────────────────────────────────────┘
```

## 本 notebook 的結構

1. 準備範例資料（建立 CSV + SQLite 原始資料）
2. 寫 `config.yaml`
3. 設定 logger
4. 實作 Extract / Transform / Load 三個函式
5. 主流程 `main()`
6. 執行 + 檢查輸出


---

## 0. 準備範例資料

實際專案的資料會從既有系統來。這裡我們用合成資料模擬：
- `orders.csv` — 訂單交易紀錄
- `customers.db` (SQLite) — 顧客主檔


In [ ]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

# 工作目錄
WORK_DIR = Path("work")
WORK_DIR.mkdir(exist_ok=True)

# 合成一份 orders CSV（故意加一些髒資料）
np.random.seed(42)
n = 200
orders = pd.DataFrame({
    "order_id": range(1001, 1001 + n),
    "customer_id": np.random.choice(["C001", "C002", "C003", "C004", "C005"], n),
    "category": np.random.choice(["3C", "服飾", "美妝", "食品"], n),
    "amount": np.random.randint(100, 5000, n).astype(float),
    "order_date": pd.date_range("2024-01-01", periods=n, freq="D").astype(str),
})
# 加一點髒資料：5% NA、2 筆重複
orders.loc[np.random.choice(n, 10, replace=False), "amount"] = np.nan
orders = pd.concat([orders, orders.head(2)], ignore_index=True)

orders.to_csv(WORK_DIR / "orders.csv", index=False)
print(f"寫出 orders.csv，共 {len(orders)} 筆")

# 合成顧客主檔（SQLite）
customers = pd.DataFrame({
    "customer_id": ["C001", "C002", "C003", "C004", "C005"],
    "name":        ["王大明", "李小華", "張美麗", "陳志強", "林雅婷"],
    "vip_level":   ["Gold", "Silver", "Gold", "Bronze", "Silver"],
    "region":      ["北部", "中部", "北部", "南部", "東部"],
})
with sqlite3.connect(WORK_DIR / "customers.db") as conn:
    customers.to_sql("customers", conn, if_exists="replace", index=False)
print("寫出 customers.db")

orders.head()

---

## 1. 設定檔（M4）

所有「會變的東西」都塞進 config — 檔名、路徑、閾值。
換環境（dev → prod）只要換 config，code 一行都不用動。


In [ ]:
config_yaml = """
pipeline:
  name: ecommerce-etl
  env: dev

extract:
  orders_csv: work/orders.csv
  customers_db: work/customers.db

transform:
  min_amount: 100         # 低於這金額視為無效訂單
  high_value_threshold: 3000

load:
  output_db: work/warehouse.db
  report_xlsx: work/monthly_report.xlsx

logging:
  level: INFO
  file: work/pipeline.log
"""

Path(WORK_DIR / "config.yaml").write_text(config_yaml, encoding="utf-8")
print("config.yaml 寫入完成")

In [ ]:
import yaml

def load_config(path: Path) -> dict:
    """讀取 YAML 設定檔"""
    with open(path, encoding="utf-8") as f:
        return yaml.safe_load(f)

config = load_config(WORK_DIR / "config.yaml")
config

---

## 2. Logger 設定（M3）

寫一個 `get_logger()` 函式，所有模組共用同一份設定。
同時輸出到 console 和檔案。


In [ ]:
import logging

def get_logger(name: str, log_file: Path, level: str = "INFO") -> logging.Logger:
    """建立一個同時寫 console + 檔案的 logger"""
    logger = logging.getLogger(name)
    logger.setLevel(level)

    # 避免重複加 handler（notebook 重跑時很常見的坑）
    if logger.handlers:
        return logger

    fmt = logging.Formatter(
        "%(asctime)s [%(levelname)s] %(name)s: %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    console = logging.StreamHandler()
    console.setFormatter(fmt)
    logger.addHandler(console)

    log_file.parent.mkdir(parents=True, exist_ok=True)
    file_handler = logging.FileHandler(log_file, encoding="utf-8")
    file_handler.setFormatter(fmt)
    logger.addHandler(file_handler)

    return logger

logger = get_logger(
    name=config["pipeline"]["name"],
    log_file=Path(config["logging"]["file"]),
    level=config["logging"]["level"],
)

logger.info("Logger 初始化完成")

---

## 3. Extract — 從 CSV + SQLite 抽資料（M1 + M2）


In [ ]:
def extract(cfg: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    """從 CSV 讀訂單、從 SQLite 讀顧客主檔"""
    logger.info("開始 Extract 階段")

    orders_path = Path(cfg["extract"]["orders_csv"])
    db_path     = Path(cfg["extract"]["customers_db"])

    logger.info("讀取 CSV：%s", orders_path)
    orders = pd.read_csv(orders_path, parse_dates=["order_date"])
    logger.info("  → 取得 %d 筆訂單", len(orders))

    logger.info("讀取 SQLite：%s", db_path)
    with sqlite3.connect(db_path) as conn:
        customers = pd.read_sql("SELECT * FROM customers", conn)
    logger.info("  → 取得 %d 筆顧客", len(customers))

    return orders, customers

orders_raw, customers_raw = extract(config)
orders_raw.head()

---

## 4. Transform — 清理、合併、計算 KPI


In [ ]:
def transform(
    orders: pd.DataFrame,
    customers: pd.DataFrame,
    cfg: dict,
) -> dict[str, pd.DataFrame]:
    """清理 + 合併 + 聚合，回傳多個結果 DataFrame"""
    logger.info("開始 Transform 階段")
    t_cfg = cfg["transform"]

    # ── 清理 ───────────────────────────
    before = len(orders)
    orders = orders.dropna(subset=["amount"]).drop_duplicates(subset=["order_id"])
    logger.info("清理後：%d → %d 筆（移除 NA + 重複）", before, len(orders))

    orders = orders[orders["amount"] >= t_cfg["min_amount"]].copy()
    logger.info("過濾低於 %s 的訂單後：%d 筆", t_cfg["min_amount"], len(orders))

    # ── 合併 ───────────────────────────
    merged = orders.merge(customers, on="customer_id", how="left")
    if merged["name"].isna().any():
        logger.warning("有 %d 筆訂單找不到對應顧客", merged["name"].isna().sum())

    # ── KPI 1：月營收 ─────────────────
    monthly_revenue = (
        merged.set_index("order_date")
        .resample("ME")["amount"]
        .sum()
        .reset_index()
        .rename(columns={"amount": "revenue"})
    )

    # ── KPI 2：VIP 等級統計 ───────────
    vip_stats = (
        merged.groupby("vip_level", as_index=False)
        .agg(
            order_count=("order_id", "count"),
            total_amount=("amount", "sum"),
            avg_amount=("amount", "mean"),
        )
        .sort_values("total_amount", ascending=False)
    )

    # ── KPI 3：高價值訂單標記 ─────────
    threshold = t_cfg["high_value_threshold"]
    merged["is_high_value"] = merged["amount"] >= threshold
    logger.info(
        "標記高價值訂單（>= %s）：%d 筆",
        threshold,
        merged["is_high_value"].sum(),
    )

    logger.info("Transform 完成")
    return {
        "clean_orders": merged,
        "monthly_revenue": monthly_revenue,
        "vip_stats": vip_stats,
    }

results = transform(orders_raw, customers_raw, config)
results["vip_stats"]

In [ ]:
results["monthly_revenue"]

---

## 5. Load — 寫回 SQLite + Excel 多分頁報表


In [ ]:
def load(results: dict[str, pd.DataFrame], cfg: dict) -> None:
    """把結果寫回 SQLite（給分析用）和 Excel（給老闆看）"""
    logger.info("開始 Load 階段")
    l_cfg = cfg["load"]

    # ── 寫 SQLite ──────────────────────
    db_path = Path(l_cfg["output_db"])
    logger.info("寫入資料倉儲：%s", db_path)
    with sqlite3.connect(db_path) as conn:
        for table_name, df in results.items():
            df.to_sql(table_name, conn, if_exists="replace", index=False)
            logger.info("  → %s (%d rows)", table_name, len(df))

    # ── 寫 Excel（多分頁）──────────────
    xlsx_path = Path(l_cfg["report_xlsx"])
    logger.info("寫入 Excel 報表：%s", xlsx_path)
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        results["monthly_revenue"].to_excel(writer, sheet_name="月營收", index=False)
        results["vip_stats"].to_excel(writer, sheet_name="VIP統計", index=False)
        results["clean_orders"].to_excel(writer, sheet_name="乾淨訂單", index=False)

    logger.info("Load 完成")

load(results, config)

---

## 6. 主流程 `main()` — 把三步驟串起來

這就是 pipeline 的總控制台。
實務上這個函式會變成 `pipeline.py` 的進入點，用 `python pipeline.py` 執行。


In [ ]:
def main(config_path: Path) -> None:
    """ETL Pipeline 主流程"""
    cfg = load_config(config_path)

    log = get_logger(
        name=cfg["pipeline"]["name"],
        log_file=Path(cfg["logging"]["file"]),
        level=cfg["logging"]["level"],
    )

    log.info("=" * 50)
    log.info("Pipeline 啟動：%s (env=%s)", cfg["pipeline"]["name"], cfg["pipeline"]["env"])
    log.info("=" * 50)

    try:
        orders, customers = extract(cfg)
        results = transform(orders, customers, cfg)
        load(results, cfg)
        log.info("Pipeline 成功完成 ✓")
    except Exception as e:
        log.exception("Pipeline 失敗：%s", e)
        raise

main(WORK_DIR / "config.yaml")

---

## 7. 驗收 — 檢查輸出


In [ ]:
# 從 warehouse.db 讀回來看看
with sqlite3.connect(config["load"]["output_db"]) as conn:
    tables = pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type='table'", conn
    )
    print("warehouse.db 裡的表：")
    print(tables)
    print()

    vip = pd.read_sql("SELECT * FROM vip_stats", conn)
    print("vip_stats：")
    print(vip)

In [ ]:
# 看 log 檔
print(Path(config["logging"]["file"]).read_text(encoding="utf-8"))

---

## 8. 延伸練習

把這個 notebook 進一步改造成**可部署的專案**：

### 🟢 基礎

1. 把所有函式（`extract` / `transform` / `load` / `main`）抽出來放到 `pipeline.py`，用 `python pipeline.py` 執行
2. 把 `config.yaml` 的路徑改用相對路徑以外的絕對路徑，觀察會發生什麼

### 🟡 進階

3. 用 `argparse` 讓 `pipeline.py` 支援 `--config` 參數，可以指定不同 config
4. 把 logger 改用 `RotatingFileHandler`，避免 log 檔無限變大
5. 加一個 `dry_run` 參數，設成 `True` 時只 extract + transform，不 load

### 🔴 挑戰

6. 為 `transform()` 寫 pytest 單元測試（給假資料、驗證輸出）
7. 用 GitHub Actions 排程每天跑一次這個 pipeline（可以用免費帳號）
8. 把 `config.yaml` 分成 `config.dev.yaml` 和 `config.prod.yaml`，用環境變數切換

---

## 恭喜 🎉

你剛剛做的事情，就是**資料工程師每天在做的事**：

- Extract：從各種來源抽資料
- Transform：清理、合併、計算
- Load：寫回資料倉儲、產出報表
- 全程 logging，出事可以追
- 所有參數靠 config，換環境不改 code
- 有 type hints，別人（或未來的你）讀得懂

這份 notebook 就是你的作品集。把它放到 GitHub，面試時直接秀給面試官看。